# SkylineGeolocation Pipeline (Colab, Resumable)

Self-contained notebook. Runs the full pipeline:
1. Clone repo if missing
2. Mount Drive + symlink data
3. Verify DB integrity
4. Generate `predicted_masks` if missing (uses U-Net model)
5. Run chunked evaluation (50 samples/chunk, checkpointed to Drive)
6. Plot results

Required Drive paths:
- `MyDrive/skyline_db.parquet`
- `MyDrive/synthetic_dataset/` (with `ground_truth.json`, `images/`, optional `masks/`)
- `MyDrive/sky_segmentation_unet_model.pth`
- `MyDrive/projectdata/dem.tif`

Resumable: state in `MyDrive/pipeline_state.json`, chunk results in `MyDrive/eval_checkpoints/`. Re-running skips completed work.

In [ ]:
import os, shutil
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
BRANCH = 'rewrite'
REPO_PARENT = str(REPO.parent)
if (REPO / 'src').exists():
    print('Repo already at', REPO)
else:
    print('Cloning repo...')
    if REPO.is_file() or REPO.is_symlink():
        REPO.unlink()
    elif REPO.is_dir():
        shutil.rmtree(REPO)
    REPO.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://github.com/pxrxp/SkylineGeolocation.git'
    get_ipython().run_line_magic('cd', REPO_PARENT)
    get_ipython().system(f'git clone --depth 1 --branch {BRANCH} {url}')
    print('Cloned to', REPO)
    get_ipython().run_line_magic('cd', str(REPO))
import sys
sys.path.insert(0, str(REPO))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -qq libgl1-mesa-glx libglib2.0-0 2>/dev/null
!pip install -q fastdtw pyarrow geopy pyprojroot segmentation-models-pytorch timm albumentations

In [ ]:
import json, sys
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
DRIVE = Path('/content/drive/MyDrive')
STATE = DRIVE / 'pipeline_state.json'

sys.path.insert(0, str(REPO))
from scripts.resume import ResumeManifest
m = ResumeManifest(STATE)
print(m.summary() if m.state else "Fresh start")


In [ ]:
import os

# Link Drive data into repo
links = [
    (DRIVE / 'skyline_db.parquet',
     REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'),
    (DRIVE / 'synthetic_dataset',
     REPO / 'data/synthetic_dataset'),
    (DRIVE / 'sky_segmentation_unet_model.pth',
     REPO / 'data/sky_segmentation_unet_model.pth'),
    (DRIVE / 'projectdata/dem.tif',
     REPO / 'data/digital_elevation_model/dem_30m.tif'),
]

for src, dst in links:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'Linked {src.name}')
    elif dst.exists():
        print(f'{dst.name}: already linked')
    else:
        print(f'MISSING: {src}')


In [ ]:
# Verify DB integrity
db_path = REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'
if db_path.exists():
    size = db_path.stat().st_size
    with open(db_path, 'rb') as f:
        f.seek(0); hdr = f.read(4)
        f.seek(-8, 2); ftr = f.read()
    if hdr != b'PAR1' or ftr[-4:] != b'PAR1':
        raise SystemExit(f'BAD: DB magic bytes missing')
    print(f'DB OK: {size / 1e6:.1f} MB, PAR1 headers intact')
    m.mark('db_verified', size_bytes=size)
    m.save()
else:
    raise SystemExit('MISSING: DB parquet not found')

In [ ]:
# Check required files
gt_path = REPO / 'data/synthetic_dataset/ground_truth.json'
model_path = REPO / 'data/sky_segmentation_unet_model.pth'
masks_dir = REPO / 'data/synthetic_dataset/predicted_masks'

if not gt_path.exists():
    raise SystemExit(f'MISSING: {gt_path} - upload ground_truth.json to Drive/MyDrive/synthetic_dataset/')
print(f'Ground truth: OK ({gt_path.stat().st_size / 1e3:.0f} KB)')

if not model_path.exists():
    raise SystemExit(f'MISSING: {model_path}')
print(f'Segmentation model: OK ({model_path.stat().st_size / 1e6:.1f} MB)')

n_masks = len(list(masks_dir.glob('*.png'))) if masks_dir.exists() else 0
print(f'Predicted masks: {n_masks}/300')
m.mark('inputs_checked', gt=gt_path.exists(), model=model_path.exists(), n_masks=n_masks)
m.save()

In [ ]:
# Generate predicted_masks if missing (skip if 300 already exist)
import torch

if n_masks >= 300:
    print('Skipping segmentation: 300 masks already present')
else:
    print(f'Need to generate {300 - n_masks} masks')
    from src.segmentation import load_segmentation_model, segment_image
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = load_segmentation_model('data/sky_segmentation_unet_model.pth', device)
    print(f'Model loaded on: {device}')

    masks_dir.mkdir(parents=True, exist_ok=True)
    images_dir = REPO / 'data/synthetic_dataset/images'
    image_paths = sorted(images_dir.glob('*.png'))
    print(f'Found {len(image_paths)} images')

    for i, img in enumerate(image_paths):
        mask_path = masks_dir / img.name
        if mask_path.exists():
            continue
        res = segment_image(model, str(img), str(mask_path), device)
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(image_paths)}: {res["status"]}', flush=True)

    n_masks = len(list(masks_dir.glob('*.png')))
    print(f'Now have {n_masks}/300 masks')

    # Sync to Drive
    import shutil
    shutil.copytree(masks_dir, DRIVE / 'synthetic_dataset/predicted_masks', dirs_exist_ok=True)
    print('Synced masks to Drive')
    m.mark('segmentation', n_masks=n_masks)
    m.save()

In [ ]:
import os
import gc
import pandas as pd
import pyarrow.parquet as pq

# Ensure DB symlink, then load metadata
db_link = REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'
db_src = DRIVE / 'skyline_db.parquet'
if not db_link.exists() and db_src.exists():
    db_link.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(db_src, db_link)
    print(f'Linked {db_src.name}')
if not db_link.exists():
    raise SystemExit(f'MISSING: {db_link} - upload skyline_db.parquet to MyDrive/')

meta = pd.read_parquet(str(db_link), columns=['lon', 'lat', 'elevation_m'])
del meta; gc.collect()
print('DB metadata loaded', flush=True)

pf = pq.ParquetFile(str(db_link))
first = next(pf.iter_batches(batch_size=1, columns=['raw_horizon_deg']))
bin_deg = 360.0 / len(first.to_pandas()['raw_horizon_deg'].iloc[0])
print(f'bin_deg={bin_deg}', flush=True)


In [ ]:
import json
import pandas as pd
from src.evaluation import run_evaluation

# Checkpointed evaluation
checkpoint_dir = DRIVE / 'eval_checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

with open('data/synthetic_dataset/ground_truth.json') as f:
    gt_all = json.load(f)
sample_ids = list(gt_all.keys())
chunk_size = 50
chunks = [sample_ids[i:i+chunk_size] for i in range(0, len(sample_ids), chunk_size)]

print(f'Total samples: {len(sample_ids)}, chunks: {len(chunks)} (size={chunk_size})')

all_dfs = []
for i, chunk_ids in enumerate(chunks):
    chunk_gt_path = checkpoint_dir / f'gt_chunk_{i}.json'
    chunk_gt = {sid: gt_all[sid] for sid in chunk_ids}
    with open(chunk_gt_path, 'w') as f:
        json.dump(chunk_gt, f)

    chunk_result_path = checkpoint_dir / f'chunk_{i}.csv'
    if chunk_result_path.exists():
        df_chunk = pd.read_csv(chunk_result_path)
        print(f'Chunk {i}: loaded from checkpoint ({len(df_chunk)} rows)', flush=True)
    else:
        print(f'Chunk {i}/{len(chunks)-1}: processing ({len(chunk_ids)} samples)', flush=True)
        df_chunk, _ = run_evaluation(
            ground_truth_path=str(chunk_gt_path),
            db_path='notebooks/02_SkylineDatabase/output/skyline_db.parquet',
            masks_dir='data/synthetic_dataset/predicted_masks',
            use_altimeter=True,
            use_compass=True,
            limit=0,
            sample_batch_size=8,
            top_k=30,
            dtw_window=15,
            correct_dist_m=500.0,
            chunk_rows=4000,
            spatial_stride=5,
        )
        df_chunk.to_csv(chunk_result_path, index=False)
        print(f'  Chunk {i}: saved to Drive ({len(df_chunk)} rows)', flush=True)

    all_dfs.append(df_chunk)

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    results_path = REPO / 'notebooks/05_SkylineMatching/output/eval_results.csv'
    results_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(results_path, index=False)
    print(f'Total eval rows: {len(df)}')
else:
    df = pd.DataFrame()
    print('No chunks processed')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot results
if len(df) > 0 and 'error_m' in df.columns:
    errors = df['error_m'].dropna()
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.hist(errors, bins=50, color='steelblue', edgecolor='white')
    plt.axvline(500, color='red', ls='--', label='500m')
    plt.xlabel('Error (m)'); plt.ylabel('Count')
    plt.title(f'Top-1 Errors (n={len(errors)})')
    plt.legend()

    plt.subplot(1, 3, 2)
    accs = [(d, (errors <= d).mean()*100) for d in [100, 500, 1000, 5000]]
    plt.bar([f'{d}m' for d,_ in accs], [a for _,a in accs], color='seagreen')
    plt.ylabel('Top-1 Accuracy (%)')
    plt.title('Accuracy Thresholds')

    plt.subplot(1, 3, 3)
    sorted_errors = np.sort(errors)
    plt.plot(sorted_errors, np.linspace(0, 1, len(sorted_errors)))
    plt.axhline(0.5, color='gray', ls=':')
    plt.axvline(500, color='red', ls='--')
    plt.xlabel('Error (m)'); plt.ylabel('Cumulative')
    plt.title('CDF')

    plt.tight_layout()
    plt.savefig(DRIVE / 'eval_results.png', dpi=150)
    plt.show()
    print(f'Median: {errors.median():.0f}m')
    print(f'Top-1@500m: {(errors <= 500).mean()*100:.1f}%')
    print(f'Top-1@100m: {(errors <= 100).mean()*100:.1f}%')
    if 'top5_ok' in df.columns:
        print(f'Top-5@500m: {df["top5_ok"].mean()*100:.1f}%')
else:
    print('No results to plot')
